# CarFix: reproduce all the results (top to bottom)

ESSEC AIDAMS · Business Data Science · Group project.
This notebook re-runs the whole project from the repository: data → index → RAG evaluation → classifier
fine-tuning and baselines → figures. Every number in the report comes from these cells.

**Before running:** `Runtime → Change runtime type → T4 GPU` (CPU works too, training is just slower), and add your
Groq key in the Colab **Secrets** panel (🔑 on the left) under the name `GROQ_API_KEY`.

In [ ]:
REPO_URL = "https://github.com/<user>/carfix"   # TODO: your repository
!git clone -q $REPO_URL carfix || (cd carfix && git pull -q)
%cd carfix
!pip -q install -e . 2>&1 | tail -1

In [ ]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
!python -c "from carfix.config import settings as s; print(s.llm_provider, s.llm_model, s.embedding_model, s.retrieval_mode, 'MIN_SCORE', s.min_score)"

## 1. Data
The processed guides are committed in `data/processed/documents.jsonl`, so the results are reproducible even if
iFixit changes its guides. Set `REFETCH = True` to download them again from the iFixit API.

In [ ]:
REFETCH = False
if REFETCH or not os.path.exists("data/processed/documents.jsonl"):
    !python scripts/fetch_ifixit.py --max-guides 150 --per-category 4
import pandas as pd
src = pd.read_csv("data/processed/sources.csv")
print(len(src), "guides from", src.category.nunique(), "car categories")
src.head(10)

## 2. Index (chunk → embed → FAISS)

In [ ]:
!python scripts/build_index.py

## 3. Try a few questions (including one with no answer in the guides)

In [ ]:
!python scripts/ask.py "How do I replace the front brake pads? What tools do I need?"
!python scripts/ask.py "How much does a timing belt replacement cost at a Toyota dealership?"

## 4. RAG evaluation
1. Calibrate the refusal threshold on the **DEV** questions only.
2. Retrieval metrics (dense vs hybrid) on **TEST**.
3. Full evaluation on **TEST**, with the "LLM without RAG" baseline.

In [ ]:
!python scripts/run_eval.py --sweep
# Copy the suggested value here, then it is used by every following cell:
os.environ["MIN_SCORE"] = "0.80"   # TODO: value printed above

In [ ]:
!python scripts/run_eval.py --retrieval-only
!python scripts/run_eval.py --baseline

In [ ]:
import glob
rag = pd.read_csv(sorted(glob.glob("eval/results/*_rag_test_*.csv"))[-1])
rag[["qid", "question", "answerable", "best_score", "refused", "grade", "citation_correct", "answer"]]

**Read the failures** (grade ≠ correct, wrong refusals, wrong citations) and explain each one in the report.

In [ ]:
rag[(rag.grade != 'correct')][['qid', 'question', 'expected_answer', 'answer', 'retrieved']]

## 5. Fine-tuning: route a car problem to the right system
The labelled data is committed in `data/ft/` (generated once with `scripts/generate_ft_data.py`, then checked by
hand). The test split and the hand-written hard set are never used for training.

In [ ]:
if not os.path.exists("data/ft/train.jsonl"):
    !python scripts/generate_ft_data.py --per-label 80
!python scripts/train_classifier.py --curve 50,150

In [ ]:
import json
rep = json.load(open("eval/results/latest_classifier.json"))
pd.DataFrame([{"model": k, "test acc": v["test"]["accuracy"], "test macro-F1": v["test"]["macro_f1"],
               "hard acc": v["hard"]["accuracy"] if v["hard"] else None} for k, v in rep["results"].items()])

## 6. Figures for the report and the summary tables

In [ ]:
!pip -q install umap-learn matplotlib
!python scripts/make_figures.py
from IPython.display import Image, Markdown, display
for f in sorted(glob.glob("reports/figures/*.png")):
    display(Image(f, width=600))
display(Markdown(open("eval/results/results.md").read()))